# AI Hub SEN JSON → 2D CTC 학습

`tools/prepare_sen.py`로 만든 학습 자료를 Colab에서 사용합니다.
현재 SYN 클립별 원본 morpheme JSON을 전체 클립 ID로 연결하고, 형태소 시작 시각순으로 정답 단어열을 구성했습니다.
원본 토큰 SYN05의 실제 화자/캐릭터 정체는 미검증입니다. 현재는 학습 실행을 확인하고 검증·테스트 정확도를 계산하지 않습니다.
2D 세 번째 값은 무시하고 유한하며 (0,0)이 아닌 x/y로 마스크를 만듭니다. 이 마스크는 검출 신뢰도가 아닙니다.

GPU 런타임을 선택한 뒤 아래 셀을 순서대로 실행하세요.

## 1. 환경과 Google Drive 연결

Drive의 `내 드라이브/SignLanguageGlass`에 새 번들의 `colab_code.zip`, `sen_training_data.zip`을 올리세요.
정규화된 NPZ를 사용하므로 원본 JSON 6만 개를 Colab에서 다시 변환하지 않습니다.
Colab에 설치된 PyTorch를 사용합니다. 로컬용 고정 requirements로 덮어쓰지 않습니다.

In [ ]:
from pathlib import Path
import json
import sys
import shutil
import tempfile
import zipfile
from datetime import datetime, timezone

import numpy as np
import torch
from google.colab import drive

drive.mount("/content/drive")
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Python:", sys.version.split()[0], "PyTorch:", torch.__version__, "device:", DEVICE)
if DEVICE == "cuda":
    print(torch.cuda.get_device_name(0))


## 2. 설정

현재 데이터에 대해 확인한 해상도는 **2048×1152**입니다. 다른 영상이나 리사이즈된 좌표에는 실제 해상도를 사용하세요.
FPS=30은 [AI Hub 키포인트 설명](https://aihub.or.kr/aihubdata/data/view.do?dataSetSn=103)에 따른 값입니다.
처음에는 20클립·3 epoch로 확인한 뒤 MAX_CLIPS=0으로 바꾸면 변환된 클립 전체를 학습합니다.

USE_PREPARED_DATA=True가 기본입니다. 원본 keypoints.zip을 사용하려면 SEN/keypoint와 SEN/morpheme을 함께 포함하고 False로 바꾸세요.

In [ ]:
DRIVE_DIR = Path("/content/drive/MyDrive/SignLanguageGlass")
IMAGE_SIZE = [2048, 1152]  # 현재 SYN 데이터의 확인된 해상도
USE_PREPARED_DATA = True
FPS = 30
MAX_CLIPS = 20  # 0: 단일 라벨 정면 클립 전체
EPOCHS = 3
BATCH = 4
SEED = 0
WORK = Path(tempfile.mkdtemp(prefix="signglass-", dir="/content"))
RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
CODE = WORK / "code"
RAW = WORK / "raw"
print("작업 위치:", WORK)

## 3. 파일을 Colab 가상 머신에 복사하고 압축 해제

JSON 수십만 개를 Drive에서 직접 읽지 않고, ZIP을 가상 머신으로 복사해 해제합니다.
런타임이 삭제되면 이 작업 폴더도 사라지므로 결과 모델은 Drive에 저장합니다.


In [ ]:
def extract_local(archive_name, target):
    archive = DRIVE_DIR / archive_name
    if not archive.is_file():
        raise FileNotFoundError(f"Drive에 파일을 업로드하세요: {archive}")
    local = WORK / archive_name
    shutil.copyfile(archive, local)
    target.mkdir(parents=True, exist_ok=False)
    with zipfile.ZipFile(local) as z:
        for member in z.infolist():
            (target / member.filename).resolve().relative_to(target.resolve())
        z.extractall(target)

extract_local("colab_code.zip", CODE)
extract_local("sen_training_data.zip" if USE_PREPARED_DATA else "keypoints.zip", RAW)
if not (CODE / "src" / "train_continuous.py").is_file():
    raise ValueError("코드 ZIP 안에는 src/와 tools/가 있어야 합니다")
sys.path.insert(0, str(CODE))
if USE_PREPARED_DATA:
    manifests = list(RAW.rglob("manifest.json"))
    if len(manifests) != 1:
        raise ValueError("학습 자료 ZIP에는 manifest.json이 하나 있어야 합니다")
    DATA_MANIFEST = manifests[0]
    print("학습 자료:", DATA_MANIFEST)
else:
    roots = [p.parent for p in RAW.rglob("SEN") if (p / "keypoint").is_dir()]
    if len(roots) != 1:
        raise ValueError("SEN/keypoint와 SEN/morpheme 폴더를 함께 압축하세요")
    DATA_ROOT = roots[0]
    print("JSON 데이터 루트:", DATA_ROOT)

## 4. 원본 라벨 연결과 변환 검사

SEN 번호만 같은 다른 촬영 클립의 라벨을 가져오지 않습니다. 클립 전체 ID와 metaData.name이 일치해야 합니다.
키포인트 시간 범위와 라벨 영상 길이가 다른 클립은 제외되며 사유가 manifest.json에 남습니다.
여러 시점을 선택해도 같은 원본 동작은 동일 group_id를 유지합니다.

In [ ]:
from tools.prepare_sen import official_annotations, convert_annotations
from src.pose_sequence import VERSION

if USE_PREPARED_DATA:
    annotations = json.loads(DATA_MANIFEST.read_text(encoding="utf-8"))
    if annotations.get("version") != VERSION or annotations.get("normalization_pending", True):
        raise ValueError("정규화가 완료된 pose2d 학습 ZIP이 필요합니다")
    if any(row.get("image_size") != IMAGE_SIZE or row.get("fps") != FPS for row in annotations["clips"]):
        raise ValueError("설정한 해상도/FPS와 학습 자료의 메타데이터가 다릅니다")
else:
    annotations = official_annotations(DATA_ROOT, size=IMAGE_SIZE, fps=FPS)
audit = annotations.get("index_audit", {})
print("선택 가능한 클립:", len(annotations["clips"]))
print("라벨 누락/오류 제외:", len(audit.get("excluded", [])))
print("좌표/시간 오류 제외:", len(annotations.get("excluded_from_conversion", [])))
print("정확한 클립 ID 라벨 연결:", annotations.get("label_alignment_verified", False))
for row in annotations["clips"][:5]:
    print(row["id"], "->", row["glosses"])

## 5. 학습 입력 준비

기본 경로에서는 선택한 NPZ만 복사합니다. 원본 JSON 경로는 검사를 통과한 클립을 정규화합니다.
원래 프레임 번호와 간격을 유지하며 3D 좌표를 합성하지 않습니다.

In [ ]:
if not isinstance(MAX_CLIPS, int) or MAX_CLIPS < 0:
    raise ValueError("MAX_CLIPS는 0(전체) 이상이어야 합니다")
PREPARED = WORK / "processed"
if USE_PREPARED_DATA:
    converted = dict(annotations)
    converted["clips"] = annotations["clips"][:MAX_CLIPS] if MAX_CLIPS else annotations["clips"]
    PREPARED.mkdir(parents=True, exist_ok=False)
    for row in converted["clips"]:
        source_file = (DATA_MANIFEST.parent / row["file"]).resolve()
        source_file.relative_to(DATA_MANIFEST.parent.resolve())
        target = (PREPARED / row["file"]).resolve()
        target.relative_to(PREPARED.resolve())
        target.parent.mkdir(parents=True, exist_ok=True)
        shutil.copyfile(source_file, target)
    converted["training_selection"] = {"available": len(annotations["clips"]), "selected": len(converted["clips"])}
    (PREPARED / "manifest.json").write_text(json.dumps(converted, ensure_ascii=False, indent=2), encoding="utf-8")
else:
    converted = convert_annotations(DATA_ROOT, annotations, PREPARED, limit=MAX_CLIPS, skip_invalid=True)
print("학습에 사용할 클립:", len(converted["clips"]), "version:", converted["version"])

## 6. CTC 학습 실행 확인

전체 선택 데이터를 학습에 사용합니다. loss는 학습 손실이며 번역 정확도가 아닙니다.
검증 데이터 없이 모델 선택이나 성능 평가를 수행하지 않습니다.
현재 한 원본 토큰/여러 시점을 무작위 분할해 높은 정확도를 보고해서는 안 됩니다.

정식 평가에는 검수된 SYN별 형태소 라벨과 실제 화자 정보, 독립 train/val/test 데이터가 필요합니다.
그때는 프로젝트의 `python -m src.train_continuous ... --device cuda` 경로를 사용하세요.

이 셀은 학습 종료 시 ctc.pt를 저장합니다. 완료 전에 런타임이 종료되면 해당 학습을 다시 실행해야 합니다.


In [ ]:
if PREPARED is None:
    print("학습 대기: 먼저 image_size를 확인하세요.")
else:
    from src.continuous_sign import CtcRecognizer, load_dataset, save_checkpoint
    from src.train_continuous import fit, feature_cache
    from src.data_sources import attribution

    manifest = json.loads((PREPARED / "manifest.json").read_text(encoding="utf-8"))
    rows, skipped = load_dataset(PREPARED / "manifest.json")
    labels = ["<blank>"] + sorted({g for row in rows for g in row["glosses"]})
    torch.manual_seed(SEED)
    if DEVICE == "cpu":
        torch.set_num_threads(2)
    model = CtcRecognizer(len(labels), feature_set="2d")
    history = fit(model, rows, labels, epochs=EPOCHS, batch=BATCH, seed=SEED,
                  device=DEVICE, cache=feature_cache(rows, "2d"))
    OUT = DRIVE_DIR / "runs" / RUN_ID
    OUT.mkdir(parents=True, exist_ok=False)
    metadata = {
        "continuous": True, "training_scope": "train_only_experiment",
        "data_sources": manifest["sources"],
        "label_alignment_verified": manifest.get("label_alignment_verified", False),
        "signer_identity_verified": manifest.get("signer_identity_verified", False),
        "mask_policy": manifest["extractor"]["mask_policy"],
        "validation_evaluated": False, "test_evaluated": False,
        "deployment_ready": False,
    }
    save_checkpoint(OUT / "ctc.pt", model.cpu(), labels, metadata=metadata)
    report = {
        **metadata, "history_loss_sum": history, "clips": [r["id"] for r in rows],
        "labels": labels, "image_size": IMAGE_SIZE, "fps": FPS,
        "epochs": EPOCHS, "batch": BATCH, "seed": SEED, "device": DEVICE,
        "versions": {"python": sys.version.split()[0], "numpy": np.__version__, "torch": str(torch.__version__)},
    }
    for name, data in [("training.json", report), ("data_audit.json", audit), ("manifest.json", manifest)]:
        (OUT / name).write_text(json.dumps(data, ensure_ascii=False, indent=2), encoding="utf-8")
    for sentence in attribution(manifest["sources"]):
        print(sentence)
    print("Drive 저장:", OUT)
    print("실행 확인 모델입니다. 검증/테스트 정확도는 계산하지 않았습니다.")

결과는 `내 드라이브/SignLanguageGlass/runs/<실행시각>/ctc.pt`에 저장됩니다.
이 파일은 연속 수어용 CTC 체크포인트입니다. 로컬 단어 ONNX 모델과 입력 규격이 다릅니다.
현재 한 SYN 원본 토큰으로 학습 손실을 확인하며, 독립 평가와 실시간 성능 측정은 별도로 진행해야 합니다.